## **1. Import Libraries**
---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

import os
import random

import cv2
import numpy as np
# from tqdm.notebook import tqdm
from tqdm import tqdm

In [20]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)  # Set seed for Python's hash function
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = True     # Optimize performance if input size is fixed
    torch.backends.cudnn.deterministic = True # Ensure reproducibility (may slow down training)

In [21]:
# For reproducibility
set_seed(42)

# Device
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_cnt = torch.cuda.device_count()
print(f"device: {device}, num device: {device_cnt}")

# Constant
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])  

device: cuda, num device: 1


## **2. Defining Models**
---

In [22]:
class TransformerNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        
        self.model = nn.Sequential(
            ConvBlock(3, 32, kernel_size=9, stride=1),
            ConvBlock(32, 64, kernel_size=3, stride=2),
            ConvBlock(64, 128, kernel_size=3, stride=2),
            ResidualBlock(128),
            ResidualBlock(128),
            ResidualBlock(128),
            ResidualBlock(128),
            ResidualBlock(128),
            ConvBlock(128, 64, kernel_size=3, upsample=True),
            ConvBlock(64, 32, kernel_size=3, upsample=True),
            ConvBlock(32, 3, kernel_size=9, stride=1, normalize=False, relu=False),
        )

    def forward(self, x):
        return self.model(x)


class ResidualBlock(torch.nn.Module):
    def __init__(self, channels):
        super().__init__()
        
        self.block = nn.Sequential(
            ConvBlock(channels, channels, kernel_size=3, stride=1, normalize=True, relu=True),
            ConvBlock(channels, channels, kernel_size=3, stride=1, normalize=True, relu=False),
        )

    def forward(self, x):
        return self.block(x) + x


class ConvBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, upsample=False, normalize=True, relu=True):
        super(ConvBlock, self).__init__()
        
        self.upsample = upsample
        self.block = nn.Sequential(
            nn.ReflectionPad2d(kernel_size // 2), 
            nn.Conv2d(in_channels, out_channels, kernel_size, stride)
        )
        self.norm = nn.InstanceNorm2d(out_channels, affine=True) if normalize else None
        self.relu = relu

    def forward(self, x):
        if self.upsample:
            x = F.interpolate(x, scale_factor=2)
        x = self.block(x)
        if self.norm is not None:
            x = self.norm(x)
        if self.relu:
            x = F.relu(x)
        return x

## **3. Utility Functions**
---

In [23]:
def preprocess_frame(frame):
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    tensor_rgb = transform(frame_rgb).unsqueeze(0)
    return tensor_rgb.to(device)

def postprocess_frame(tensor):
    img_np  = tensor.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
    img_np  = np.clip(img_np, 0, 1)
    img_np  = (img_np * 255).astype(np.uint8)
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    
    return img_bgr

def denormalize(tensors):
    for c in range(3):
        tensors[:, c].mul_(std[c]).add_(mean[c])
    return tensors

## **4. Inference**
---

In [24]:
def inference_video(content_video, style_name, checkpoint_model, save_path):
    os.makedirs(os.path.join(save_path, "results"), exist_ok=True)

    # Load model
    transformer = TransformerNet().to(device)
    transformer.load_state_dict(torch.load(checkpoint_model, weights_only=True))
    transformer.eval()

    # Video attributes
    cap  = cv2.VideoCapture(content_video)
    fps  = int(cap.get(cv2.CAP_PROP_FPS))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video attributes: {fps} FPS, {w} x {h}")

    # Writer for saving video
    output_path = os.path.join(save_path, "results", f"{style_name}_video.mp4")
    fourcc      = cv2.VideoWriter_fourcc(*'mp4v') # Codec
    out         = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    # Inference
    with tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc='Stylizing Video', unit='frame') as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
    
            tensor_rgb = preprocess_frame(frame)
            with torch.no_grad():
                stylized_tensor = transformer(tensor_rgb)
            stylized_tensor = denormalize(stylized_tensor).cpu()
            stylized_frame  = postprocess_frame(stylized_tensor)
            out.write(stylized_frame)

            # Update pbar
            pbar.update(1)


    # Release
    cap.release()
    out.release()

    print('Done!')

In [ ]:
ckpt        = './weights/best_model-Post_Impressionism.pth'
STYLE       = os.path.basename(ckpt).split('.')[0].split('-')[1]
CONTENT_VID = './resources/videos/content_20s.mp4'

inference_video(
    content_video=CONTENT_VID, 
    style_name=STYLE, 
    checkpoint_model=ckpt, 
    save_path='./'
)

Video attributes: 30 FPS, 1920 x 1080


Stylizing Video:   0%|          | 0/606 [00:00<?, ?frame/s]

Done!


In [ ]:
ckpt        = './weights/best_model-Cubism.pth'
STYLE       = os.path.basename(ckpt).split('.')[0].split('-')[1]
CONTENT_VID = './resources/videos/content_20s.mp4'

inference_video(
    content_video=CONTENT_VID, 
    style_name=STYLE, 
    checkpoint_model=ckpt, 
    save_path='./'
)

Video attributes: 30 FPS, 1920 x 1080


Stylizing Video:   0%|          | 0/606 [00:00<?, ?frame/s]

Done!


In [ ]:
ckpt        = './weights/best_model-Abstract_Expressionism.pth'
STYLE       = os.path.basename(ckpt).split('.')[0].split('-')[1]
CONTENT_VID = './resources/videos/content_20s.mp4'

inference_video(
    content_video=CONTENT_VID, 
    style_name=STYLE, 
    checkpoint_model=ckpt, 
    save_path='./'
)

Video attributes: 30 FPS, 1920 x 1080


Stylizing Video:   0%|          | 0/606 [00:00<?, ?frame/s]

Done!


In [ ]:
ckpt        = './weights/best_model-Digital_Painting.pth'
STYLE       = os.path.basename(ckpt).split('.')[0].split('-')[1]
CONTENT_VID = './resources/videos/content_20s.mp4'

inference_video(
    content_video=CONTENT_VID, 
    style_name=STYLE, 
    checkpoint_model=ckpt, 
    save_path='./'
)

Video attributes: 30 FPS, 1920 x 1080


Stylizing Video:   0%|          | 0/606 [00:00<?, ?frame/s]

Done!
